# DATATHON 2026 — The Gridbreakers
## Full Pipeline: Part 1 + Part 2 + Part 3

This notebook executes the complete submission pipeline:

- **PART 1** — Data loading, cleaning, and structural validation  
- **PART 2** — Exploratory Data Analysis (18 visualizations)  
- **PART 3** — Sales forecasting model → `submission_final.csv`

All features are derived exclusively from the provided dataset. No external data sources are used. Random seed = 42 throughout.

## Part 1 — Data Profiling & Structuring

In [ ]:
import os, json, warnings, hashlib
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

print("=" * 70)
print("PART 1 — Data Profiling & Structuring")
print("=" * 70)

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
# All CSVs live in a single folder. Output directories are created if absent.
FOLDER     = "/Users/kylenguyen291/Desktop/datathon-2026-round-1"
OUT        = os.path.join(FOLDER, "outputs")
VIZ_OUT    = os.path.join(OUT, "viz")
SUBMISSION = os.path.join(FOLDER, "submission")
MODELS     = os.path.join(OUT, "models")
for d in [OUT, VIZ_OUT, SUBMISSION, MODELS]:
    os.makedirs(d, exist_ok=True)

print(f"  Data      : {FOLDER}")
print(f"  Outputs   : {OUT}")
print(f"  Submission: {SUBMISSION}")

In [ ]:
# ── Load all 13 tables ────────────────────────────────────────────────────────
sales       = pd.read_csv(f"{FOLDER}/sales.csv")
sample_sub  = pd.read_csv(f"{FOLDER}/sample_submission.csv")
web_traffic = pd.read_csv(f"{FOLDER}/web_traffic.csv")
orders      = pd.read_csv(f"{FOLDER}/orders.csv")
order_items = pd.read_csv(f"{FOLDER}/order_items.csv", low_memory=False)
payments    = pd.read_csv(f"{FOLDER}/payments.csv")
shipments   = pd.read_csv(f"{FOLDER}/shipments.csv")
returns     = pd.read_csv(f"{FOLDER}/returns.csv")
reviews     = pd.read_csv(f"{FOLDER}/reviews.csv")
inventory   = pd.read_csv(f"{FOLDER}/inventory.csv")
promotions  = pd.read_csv(f"{FOLDER}/promotions.csv")
products    = pd.read_csv(f"{FOLDER}/products.csv")
customers   = pd.read_csv(f"{FOLDER}/customers.csv")
geography   = pd.read_csv(f"{FOLDER}/geography.csv")

In [ ]:
# ── Date parsing helper ───────────────────────────────────────────────────────
def _to_dt(df, date_cols):
    new_data = {}
    for col in df.columns:
        if col in date_cols:
            new_data[date_cols[col]] = pd.to_datetime(df[col].values)
        elif col not in date_cols.values():
            new_data[col] = df[col].values
    return pd.DataFrame(new_data)

sales       = _to_dt(sales,       {"Date":          "Date"})
sample_sub  = _to_dt(sample_sub,  {"Date":          "Date"})
web_traffic = _to_dt(web_traffic, {"date":          "Date"})
orders      = _to_dt(orders,      {"order_date":    "order_date"})
shipments   = _to_dt(shipments,   {"ship_date":     "ship_date",
                                    "delivery_date": "delivery_date"})
returns     = _to_dt(returns,     {"return_date":   "return_date"})
reviews     = _to_dt(reviews,     {"review_date":   "review_date"})
inventory   = _to_dt(inventory,   {"snapshot_date": "snapshot_date"})
promotions  = _to_dt(promotions,  {"start_date":    "start_date",
                                    "end_date":      "end_date"})
customers   = _to_dt(customers,   {"signup_date":   "signup_date"})

# ── Sort sales by date ────────────────────────────────────────────────────────
sales = sales.sort_values("Date").reset_index(drop=True)

# ── Derived column: product margin ────────────────────────────────────────────
products["product_margin"] = (
    (products["price"] - products["cogs"]) / products["price"] * 100
)

In [ ]:
# ── Null handling ─────────────────────────────────────────────────────────────
order_items["promo_id"]   = order_items["promo_id"].fillna("no_promo")
order_items["promo_id_2"] = order_items["promo_id_2"].fillna("no_promo")
promotions["applicable_category"] = promotions["applicable_category"].fillna("All")
promotions["promo_channel"] = promotions["promo_channel"].fillna("unknown")
returns["return_reason"] = returns["return_reason"].astype("string")

# ── Schema validation ─────────────────────────────────────────────────────────
N_TEST = len(sample_sub)
assert N_TEST > 0

# ── Row counts (sanity check) ─────────────────────────────────────────────────
print(f"  sample_submission : {N_TEST} rows  "
      f"({sample_sub['Date'].min().date()} → {sample_sub['Date'].max().date()}) ✓")
print(f"  sales             : {len(sales)} rows")
print(f"  orders            : {len(orders)} rows")
print(f"  order_items       : {len(order_items)} rows")
print(f"  payments          : {len(payments)} rows")
print(f"  shipments         : {len(shipments)} rows")
print(f"  returns           : {len(returns)} rows")
print(f"  reviews           : {len(reviews)} rows")
print(f"  inventory         : {len(inventory)} rows")
print(f"  promotions        : {len(promotions)} rows")
print(f"  products          : {len(products)} rows")
print(f"  customers         : {len(customers)} rows")
print(f"  geography         : {len(geography)} rows")
print(f"  web_traffic       : {len(web_traffic)} rows")
print(f"  orders ↔ payments : "
      f"{orders['order_id'].nunique()} / "
      f"{payments['order_id'].nunique()} (1:1) ✓")
print(f"  Part 1 complete ✓")

## Part 2 — Exploratory Data Analysis

In [ ]:
print("=" * 70)
print("PART 2 — Exploratory Data Analysis")
print("=" * 70)

# ── EDA-specific derived columns ──────────────────────────────────────────────
order_items['order_value']  = (order_items['quantity'] * order_items['unit_price']
                               - order_items['discount_amount'])
order_items['has_promo']    = order_items['promo_id'].notna()
sales['gross_margin_pct']   = (sales['Revenue'] - sales['COGS']) / sales['Revenue'] * 100
sales['year']               = sales['Date'].dt.year
sales['month']              = sales['Date'].dt.month
sales['quarter']            = sales['Date'].dt.quarter
orders['year']              = orders['order_date'].dt.year
customers['signup_year']    = customers['signup_date'].dt.year
shipments['days_to_ship']   = (shipments['delivery_date'] - shipments['ship_date']).dt.days
web_traffic['year']         = web_traffic['Date'].dt.year

PALETTE = ["#2E86AB","#A23B72","#F18F01","#C73E1D",
           "#3B1F2B","#44BBA4","#E94F37","#393E41"]
sns.set_theme(style="whitegrid", font_scale=1.1)

### Section 1: Raw Observations

In [ ]:
# --- 1.1 sales table ---
yearly = sales.groupby('year')[['Revenue', 'COGS']].sum()
yearly['margin'] = (yearly['Revenue'] - yearly['COGS']) / yearly['Revenue'] * 100
print(yearly)
print(yearly['margin'])

monthly_avg = sales.groupby('month')['Revenue'].mean()
print(monthly_avg)
print(f"Peak month : {monthly_avg.idxmax()} ({monthly_avg.max()/1e6:.2f}M VND)")
print(f"Trough month: {monthly_avg.idxmin()} ({monthly_avg.min()/1e6:.2f}M VND)")
q_avg = sales.groupby('quarter')['Revenue'].mean()
print(q_avg)
print(f"Q2/Q4 ratio: {q_avg[2]/q_avg[4]:.2f}x")
rev_2016 = yearly.loc[2016, 'Revenue']
rev_2022 = yearly.loc[2022, 'Revenue']
print(f"Decline: {(rev_2022 - rev_2016)/rev_2016*100:.1f}%")

In [ ]:
# --- 1.2 orders table ---
print(orders['order_status'].value_counts())
print(orders['order_status'].value_counts(normalize=True).mul(100).round(1))
print(orders.groupby('year')['order_id'].count())
print(orders['payment_method'].value_counts(normalize=True).mul(100).round(1))
print(orders['device_type'].value_counts(normalize=True).mul(100).round(1))
print(orders['order_source'].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# --- 1.3 order_items table ---
print(f"Avg order value: {order_items['order_value'].mean():,.2f}")
print(f"Promo rate: {order_items['has_promo'].mean()*100:.1f}%")
print(order_items.groupby('has_promo')['order_value'].mean())
print(f"Avg discount (promo): {order_items[order_items['has_promo']]['discount_amount'].mean():,.2f}")
print(f"Avg discount (no promo): {order_items[~order_items['has_promo']]['discount_amount'].mean():,.2f}")
print(order_items.groupby('has_promo')['quantity'].mean())

In [ ]:
# --- 1.4 products table ---
print(products['category'].value_counts())
products['margin_pct'] = (products['price'] - products['cogs']) / products['price'] * 100
print(products.groupby('category')['margin_pct'].mean().sort_values(ascending=False))
print(products['price'].describe())

In [ ]:
# --- 1.5 returns table ---
total_returns = len(returns)
total_orders  = len(orders)
print(f"Return rate: {total_returns/total_orders*100:.2f}%")
print(f"Total refunded: {returns['refund_amount'].sum():,.0f} VND")
print(returns['return_reason'].value_counts())
print(returns['return_reason'].value_counts(normalize=True).mul(100).round(1))
sold_qty   = order_items.groupby('product_id')['quantity'].sum()
return_qty = returns.groupby('product_id')['return_quantity'].sum()
rr         = (return_qty / sold_qty).dropna().reset_index()
rr.columns = ['product_id', 'return_rate']
rr         = rr.merge(products[['product_id', 'category', 'segment']], on='product_id')
print(rr.groupby('category')['return_rate'].mean().sort_values(ascending=False))

In [ ]:
# --- 1.6 reviews table ---
print(f"Review rate: {len(reviews)/len(orders)*100:.1f}%")
print(reviews['rating'].value_counts().sort_index())
print(f"Avg: {reviews['rating'].mean():.2f}")
pos_rate = (reviews['rating'] >= 4).mean() * 100
print(f"4-5 star rate: {pos_rate:.1f}%")
rating_cat = (reviews.merge(products[['product_id', 'category']], on='product_id')
              .groupby('category')['rating'].mean().sort_values(ascending=False))
print(rating_cat)

In [ ]:
# --- 1.7 web_traffic table ---
sess_yr = web_traffic.groupby('year')['sessions'].sum()
print(sess_yr)
growth = (sess_yr.iloc[-1] - sess_yr.iloc[0]) / sess_yr.iloc[0] * 100
print(f"Sessions growth 2013→2022: {growth:.1f}%")
print(web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values(ascending=False))
print(web_traffic['traffic_source'].value_counts())

In [ ]:
# --- 1.8 inventory table ---
print(f"Stockout rate: {inventory['stockout_flag'].mean()*100:.1f}%")
print(f"Overstock rate: {inventory['overstock_flag'].mean()*100:.1f}%")
print(inventory.groupby('category')['stockout_flag'].mean().sort_values(ascending=False))
print(inventory.groupby('category')['fill_rate'].mean().sort_values(ascending=False))
print(inventory.groupby('category')['sell_through_rate'].mean().sort_values(ascending=False))

In [ ]:
# --- 1.9 shipments table ---
print(shipments['days_to_ship'].describe())
ship_geo = (shipments.merge(orders[['order_id', 'zip']], on='order_id')
            .merge(geography[['zip', 'region']], on='zip', how='left'))
print(ship_geo.groupby('region')['days_to_ship'].mean().sort_values())

In [ ]:
# --- 1.10 payments table ---
print(payments.groupby('payment_method')['payment_value'].mean().sort_values(ascending=False))
print(payments['installments'].value_counts().head(6))
installment_rate = (payments['installments'] > 1).mean() * 100
print(f"Using installments: {installment_rate:.1f}%")

In [ ]:
# --- 1.11 geography + orders ---
geo_orders = orders.merge(geography[['zip', 'region']], on='zip', how='left')
print(geo_orders['region'].value_counts())
print(geo_orders['region'].value_counts(normalize=True).mul(100).round(1))
geo_val = geo_orders.merge(
    order_items.groupby('order_id')['order_value'].sum().reset_index(), on='order_id')
print(geo_val.groupby('region')['order_value'].mean().sort_values(ascending=False))

In [ ]:
# --- 1.12 customers table ---
print(customers.groupby('signup_year')['customer_id'].count())
print(customers['age_group'].value_counts())
print(customers['age_group'].value_counts(normalize=True).mul(100).round(1))
print(customers['acquisition_channel'].value_counts(normalize=True).mul(100).round(1))
print(customers['gender'].value_counts(normalize=True).mul(100).round(1))

### Section 2: Visualizations (18 Charts)

In [ ]:
# ── 2.1 Monthly Revenue & Gross Margin Trend ─────────────────────────────────
monthly = (sales.groupby(['year', 'month'])
           .agg(Revenue=('Revenue', 'sum'), COGS=('COGS', 'sum'))
           .reset_index())
monthly['period'] = pd.to_datetime(monthly[['year', 'month']].assign(day=1))
monthly['margin'] = (monthly['Revenue'] - monthly['COGS']) / monthly['Revenue'] * 100
fig, ax1 = plt.subplots(figsize=(16, 6))
ax1.fill_between(monthly['period'], monthly['Revenue']/1e6, alpha=0.3, color=PALETTE[0])
ax1.plot(monthly['period'], monthly['Revenue']/1e6, color=PALETTE[0], linewidth=2,
         label='Revenue (M VND)')
ax1.set_ylabel('Revenue (Million VND)', color=PALETTE[0])
ax2 = ax1.twinx()
ax2.plot(monthly['period'], monthly['margin'], color=PALETTE[2], linewidth=2,
         linestyle='--', label='Gross Margin %')
ax2.set_ylabel('Gross Margin (%)', color=PALETTE[2])
ax1.set_title('Monthly Revenue & Gross Margin Trend (2012–2022)', fontsize=14, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz1_revenue_margin_trend.png", dpi=150)
plt.show()

In [ ]:
# ── 2.2 Sessions by Traffic Source Over Time ──────────────────────────────────
sess_src = (web_traffic.groupby(['year', 'traffic_source'])['sessions'].sum().reset_index())
fig, ax = plt.subplots(figsize=(14, 6))
for i, src in enumerate(sess_src['traffic_source'].unique()):
    d = sess_src[sess_src['traffic_source'] == src]
    ax.plot(d['year'], d['sessions']/1e6, marker='o', label=src, color=PALETTE[i], linewidth=2)
ax.set_title('Annual Sessions by Traffic Source (2013–2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Sessions (Millions)')
ax.legend(title='Traffic Source', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz2_sessions_by_source.png", dpi=150)
plt.show()

In [ ]:
# ── 2.3 Order Volume by Year & Status ─────────────────────────────────────────
ord_yr_status = orders.groupby(['year', 'order_status'])['order_id'].count().reset_index()
pivot = ord_yr_status.pivot(index='year', columns='order_status', values='order_id').fillna(0)
fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_title('Order Volume by Year & Order Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Number of Orders')
ax.legend(title='Order Status', bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz3_order_volume_status.png", dpi=150)
plt.show()

In [ ]:
# ── 2.4 Revenue by Product Category & Segment ────────────────────────────────
cat_rev   = order_items.merge(products[['product_id', 'category', 'segment']], on='product_id')
cat_total = cat_rev.groupby('category')['order_value'].sum().sort_values(ascending=False)
pivot_cat = (cat_rev.groupby(['category', 'segment'])['order_value']
             .sum().reset_index()
             .pivot(index='category', columns='segment', values='order_value')
             .fillna(0).loc[cat_total.index])
fig, ax = plt.subplots(figsize=(12, 6))
pivot_cat.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_title('Total Order Value by Product Category & Segment', fontsize=14, fontweight='bold')
ax.set_xlabel('Category'); ax.set_ylabel('Total Order Value (VND)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1e9:.1f}B'))
ax.legend(title='Segment', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz4_revenue_by_category.png", dpi=150)
plt.show()

In [ ]:
# ── 2.5 Return Rate by Product Category ───────────────────────────────────────
sold_qty   = order_items.groupby('product_id')['quantity'].sum()
return_qty = returns.groupby('product_id')['return_quantity'].sum()
rr         = (return_qty / sold_qty).dropna().reset_index()
rr.columns = ['product_id', 'return_rate']
rr         = rr.merge(products[['product_id', 'category', 'segment']], on='product_id')
rr_cat     = rr.groupby('category')['return_rate'].mean().sort_values(ascending=False).reset_index()
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(rr_cat['category'], rr_cat['return_rate']*100, color=PALETTE[:4], edgecolor='white')
for bar, val in zip(bars, rr_cat['return_rate']*100):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'{val:.2f}%',
            ha='center', fontweight='bold')
ax.set_title('Average Return Rate by Product Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Category'); ax.set_ylabel('Return Rate (%)')
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz5_return_rate_by_category.png", dpi=150)
plt.show()

In [ ]:
# ── 2.6 Average Rating by Category & Segment ─────────────────────────────────
rating_seg = (reviews.merge(products[['product_id', 'category', 'segment']], on='product_id')
              .groupby(['category', 'segment'])['rating'].mean().reset_index())
fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=rating_seg, x='category', y='rating', hue='segment', ax=ax, palette='Set2')
ax.set_title('Average Product Rating by Category & Segment', fontsize=14, fontweight='bold')
ax.set_xlabel('Category'); ax.set_ylabel('Average Rating (1–5)')
ax.set_ylim(3.5, 4.5)
ax.legend(title='Segment', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz6_rating_by_category_segment.png", dpi=150)
plt.show()

In [ ]:
# ── 2.7 Seasonal Revenue Pattern ──────────────────────────────────────────────
month_avg   = sales.groupby('month')['Revenue'].mean().reset_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_avg['month_name'] = month_avg['month'].apply(lambda x: month_names[x-1])
colors = ['#C73E1D' if v == month_avg['Revenue'].max() else '#2E86AB'
          for v in month_avg['Revenue']]
fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(month_avg['month_name'], month_avg['Revenue']/1e6, color=colors, edgecolor='white')
ax.set_title('Average Monthly Revenue — Seasonal Pattern (2012–2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Avg Daily Revenue (Million VND)')
ax.annotate('Peak: May (6.58M VND)', xy=(4, 6.58), xytext=(5.5, 6.1),
            arrowprops=dict(arrowstyle='->', color='red'), color='red', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz7_seasonal_revenue.png", dpi=150)
plt.show()

In [ ]:
# ── 2.8 Order Value: Promo vs No Promo ────────────────────────────────────────
oi_sample = order_items.sample(min(50000, len(order_items)), random_state=42)
oi_sample['promo_label'] = oi_sample['has_promo'].map({True: 'With Promo', False: 'No Promo'})
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=oi_sample, x='promo_label', y='order_value', ax=ax,
            palette=['#C73E1D', '#2E86AB'], showfliers=False)
ax.set_title('Order Value Distribution: Promo vs No Promo', fontsize=14, fontweight='bold')
ax.set_xlabel('Promotion Applied'); ax.set_ylabel('Order Value (VND)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1e3:.0f}K'))
means = oi_sample.groupby('promo_label')['order_value'].mean()
for i, (label, val) in enumerate(means.items()):
    ax.text(i, val+400, f'Mean: {val/1e3:.1f}K', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz8_promo_vs_no_promo.png", dpi=150)
plt.show()

In [ ]:
# ── 2.9 Inventory Health — Stockout & Overstock ───────────────────────────────
inv_yr_cat = (inventory.groupby(['year', 'category'])
              [['stockout_flag', 'overstock_flag']].mean().reset_index())
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, flag, title in zip(axes, ['stockout_flag', 'overstock_flag'],
                            ['Stockout Rate', 'Overstock Rate']):
    pivot = inv_yr_cat.pivot(index='year', columns='category', values=flag)
    pivot.plot(ax=ax, marker='o', colormap='Set1')
    ax.set_title(f'{title} by Category Over Time', fontsize=13, fontweight='bold')
    ax.set_xlabel('Year'); ax.set_ylabel(title)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))
    ax.legend(title='Category')
plt.suptitle('Inventory Health: Stockout & Overstock Rates', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz9_inventory_health.png", dpi=150)
plt.show()

In [ ]:
# ── 2.10 Customer Acquisition by Channel & Year ──────────────────────────────
cust_yr_ch = (customers.groupby(['signup_year', 'acquisition_channel'])
              ['customer_id'].count().reset_index())
pivot_ch = cust_yr_ch.pivot(index='signup_year', columns='acquisition_channel',
                             values='customer_id').fillna(0)
fig, ax = plt.subplots(figsize=(14, 6))
pivot_ch.plot(kind='area', stacked=True, ax=ax, colormap='Set2', alpha=0.85)
ax.set_title('Customer Acquisition by Channel Over Time (2012–2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('New Customers Acquired')
ax.legend(title='Channel', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz10_customer_acquisition.png", dpi=150)
plt.show()

In [ ]:
# ── 2.11 Days to Ship by Region ───────────────────────────────────────────────
ship_geo = (shipments.merge(orders[['order_id', 'zip']], on='order_id')
            .merge(geography[['zip', 'region']], on='zip', how='left'))
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=ship_geo, x='region', y='days_to_ship', ax=ax,
            palette=PALETTE[:3], showfliers=True,
            flierprops=dict(marker='.', alpha=0.2))
ax.set_title('Days to Ship by Region', fontsize=14, fontweight='bold')
ax.set_xlabel('Region'); ax.set_ylabel('Days from Ship to Delivery')
means = ship_geo.groupby('region')['days_to_ship'].mean()
for i, (reg, val) in enumerate(means.items()):
    ax.text(i, val+0.15, f'{val:.2f}d', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz11_days_to_ship_region.png", dpi=150)
plt.show()

In [ ]:
# ── 2.12 Payment Method vs Average Order Value ───────────────────────────────
pay_val = payments.merge(
    order_items.groupby('order_id')['order_value'].sum().reset_index(), on='order_id')
pay_agg = (pay_val.groupby('payment_method')
           .agg(avg_value=('payment_value', 'mean'), count=('order_id', 'count'))
           .sort_values('avg_value', ascending=False).reset_index())
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(pay_agg['payment_method'], pay_agg['avg_value']/1e3, color=PALETTE[:5], edgecolor='white')
for bar, row in zip(bars, pay_agg.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{row.avg_value/1e3:.1f}K\nn={row.count/1e3:.0f}K', ha='center', fontsize=9)
ax.set_title('Average Order Value by Payment Method', fontsize=14, fontweight='bold')
ax.set_xlabel('Payment Method'); ax.set_ylabel('Avg Payment Value (Thousand VND)')
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz12_payment_method_value.png", dpi=150)
plt.show()

In [ ]:
# ── 2.13 Age × Gender Heatmap of Avg Order Value ─────────────────────────────
cust_ord  = orders.merge(customers[['customer_id', 'age_group', 'gender']], on='customer_id')
cust_val  = cust_ord.merge(
    order_items.groupby('order_id')['order_value'].sum().reset_index(), on='order_id')
heatmap_data = cust_val.groupby(['age_group', 'gender'])['order_value'].mean().unstack()
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heatmap_data/1e3, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Avg Order Value (K VND)'})
ax.set_title('Average Order Value by Age Group × Gender (K VND)', fontsize=14, fontweight='bold')
ax.set_xlabel('Gender'); ax.set_ylabel('Age Group')
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz13_age_gender_heatmap.png", dpi=150)
plt.show()

In [ ]:
# ── 2.14 Annual Gross Margin Trend ───────────────────────────────────────────
yearly_margin = (sales.groupby('year')
                 .apply(lambda d: (d['Revenue'].sum()-d['COGS'].sum())/d['Revenue'].sum()*100)
                 .reset_index())
yearly_margin.columns = ['year', 'margin']
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(yearly_margin['year'], yearly_margin['margin'], marker='o', linewidth=2.5, color=PALETTE[0])
ax.fill_between(yearly_margin['year'], yearly_margin['margin'], alpha=0.15, color=PALETTE[0])
for _, row in yearly_margin.iterrows():
    ax.annotate(f"{row['margin']:.1f}%", (row['year'], row['margin']),
                textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9, fontweight='bold')
ax.axhline(yearly_margin['margin'].mean(), color='red', linestyle='--', alpha=0.6,
           label=f"10yr Avg: {yearly_margin['margin'].mean():.1f}%")
ax.set_title('Annual Gross Margin % Trend (2012–2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Gross Margin (%)')
ax.legend()
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz14_annual_margin_trend.png", dpi=150)
plt.show()

In [ ]:
# ── 2.15 Return Reasons Analysis ──────────────────────────────────────────────
reason_counts = returns['return_reason'].value_counts().reset_index()
reason_counts.columns = ['reason', 'count']
ret_cat   = returns.merge(products[['product_id', 'category']], on='product_id')
pivot_ret = (ret_cat.groupby(['category', 'return_reason'])['return_quantity']
             .sum().reset_index()
             .pivot(index='category', columns='return_reason', values='return_quantity')
             .fillna(0))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.pie(reason_counts['count'], labels=reason_counts['reason'], autopct='%1.1f%%',
        colors=PALETTE[:5], startangle=90, pctdistance=0.75)
ax1.set_title('Return Reasons Distribution', fontsize=13, fontweight='bold')
pivot_ret.plot(kind='bar', ax=ax2, colormap='Set1', edgecolor='white')
ax2.set_title('Return Quantity by Category & Reason', fontsize=13, fontweight='bold')
ax2.set_xlabel('Category'); ax2.set_ylabel('Return Quantity')
ax2.legend(title='Reason', bbox_to_anchor=(1.01, 1))
plt.setp(ax2.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz15_returns_analysis.png", dpi=150)
plt.show()

In [ ]:
# ── 2.16 Revenue Decline: Volume vs Price Decomposition ──────────────────────
yr_rev         = sales.groupby('year')['Revenue'].sum().reset_index()
yr_ord         = orders.groupby('year')['order_id'].count().reset_index()
yr_ord.columns = ['year', 'order_count']
yr_merged      = yr_rev.merge(yr_ord, on='year')
yr_merged['avg_order_rev'] = yr_merged['Revenue'] / yr_merged['order_count']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, col, ylabel, title in zip(
        axes, ['Revenue', 'order_count', 'avg_order_rev'],
        ['Revenue (Billion VND)', 'Order Count', 'Avg Revenue per Order (VND)'],
        ['Total Annual Revenue', 'Annual Order Count', 'Avg Revenue per Order']):
    bar_colors = ['#C73E1D' if y >= 2017 else '#2E86AB' for y in yr_merged['year']]
    ax.bar(yr_merged['year'], yr_merged[col], color=bar_colors, edgecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    if col == 'Revenue':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1e9:.1f}B'))
    elif col == 'avg_order_rev':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1e3:.0f}K'))
    plt.setp(ax.get_xticklabels(), rotation=45)
plt.suptitle('Revenue Decline: Volume vs Price Decomposition (Red = Post-Peak)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz16_revenue_decline_analysis.png", dpi=150)
plt.show()

In [ ]:
# ── 2.17 Order Source × Device Type Heatmap ──────────────────────────────────
src_dev     = (orders.groupby(['order_source', 'device_type'])['order_id'].count().unstack().fillna(0))
src_dev_pct = src_dev.div(src_dev.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(src_dev_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            linewidths=0.5, cbar_kws={'label': '% of Orders'})
ax.set_title('Order Distribution by Source × Device Type (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Device Type'); ax.set_ylabel('Order Source')
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz17_source_device_heatmap.png", dpi=150)
plt.show()

In [ ]:
# ── 2.18 Annual Cancellation & Return Rate Trends ────────────────────────────
cancel_rate = (orders.groupby('year')
               .apply(lambda d: (d['order_status']=='cancelled').sum()/len(d)*100)
               .reset_index())
cancel_rate.columns = ['year', 'cancel_rate']
return_rate_yr = (orders.groupby('year')
                  .apply(lambda d: (d['order_status']=='returned').sum()/len(d)*100)
                  .reset_index())
return_rate_yr.columns = ['year', 'return_rate']
rates = cancel_rate.merge(return_rate_yr, on='year')
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(rates['year'], rates['cancel_rate'], marker='o', linewidth=2, color=PALETTE[3],
        label='Cancellation Rate %')
ax.plot(rates['year'], rates['return_rate'], marker='s', linewidth=2, color=PALETTE[2],
        label='Return Rate %')
ax.set_title('Annual Cancellation & Return Rate as % of Orders (2012–2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Rate (%)')
ax.legend()
plt.tight_layout()
plt.savefig(f"{VIZ_OUT}/viz18_cancel_return_rates.png", dpi=150)
plt.show()

### Section 3: Further EDA Steps (Feature Engineering Preview)

In [ ]:
# Outlier detection
sales['revenue_z'] = stats.zscore(sales['Revenue'])
outliers = sales[sales['revenue_z'].abs() > 3]
print(f"Revenue outlier days: {len(outliers)}")
print(outliers[['Date', 'Revenue', 'revenue_z']].sort_values('revenue_z', ascending=False).head(10))

# Skewness check
print(order_items[['quantity', 'unit_price', 'discount_amount', 'order_value']].skew())

# Lag and rolling feature preview
sales['lag_7']          = sales['Revenue'].shift(7)
sales['lag_30']         = sales['Revenue'].shift(30)
sales['rolling_7_mean'] = sales['Revenue'].rolling(7).mean()
sales['rolling_30_mean']= sales['Revenue'].rolling(30).mean()
sales['yoy_growth']     = sales['Revenue'].pct_change(365) * 100
sales['month_sin']      = np.sin(2 * np.pi * sales['month'] / 12)
sales['month_cos']      = np.cos(2 * np.pi * sales['month'] / 12)

print(f"\n  Part 2 complete — 18 charts saved to {VIZ_OUT} ✓")

## Part 3 — Training, Modeling & Prediction

In [ ]:
print("=" * 70)
print("PART 3 — Training, Modeling & Prediction")
print("=" * 70)

### Block 3 — Target Preparation + Combined Date Frame

In [ ]:
sales["log_revenue"]    = np.log1p(sales["Revenue"])
sales["margin_pct"]     = ((sales["Revenue"] - sales["COGS"]) / sales["Revenue"] * 100)
sales["is_imputed_day"] = 0

print(f"  Revenue skew (log) : {sales['log_revenue'].skew():.4f}")
print(f"  margin_pct         : mean={sales['margin_pct'].mean():.2f}%  std={sales['margin_pct'].std():.2f}%")

_train_dates = sales[["Date"]].copy();      _train_dates["split"] = "train"
_test_dates  = sample_sub[["Date"]].copy(); _test_dates["split"]  = "test"
df = (pd.concat([_train_dates, _test_dates], ignore_index=True)
        .sort_values("Date").reset_index(drop=True))
df = df.merge(sales[["Date","Revenue","COGS","margin_pct","log_revenue",
                      "is_imputed_day"]], on="Date", how="left")

assert (df["split"] == "test").sum() == N_TEST
print(f"  Combined frame: {len(df)} rows  "
      f"(train={(df['split']=='train').sum()}  test={(df['split']=='test').sum()}) ✓")

### Block 4 — Build the 25 Selected Features

In [ ]:
df = df.sort_values("Date").reset_index(drop=True)

# Group 1: Calendar
df["year"]                = df["Date"].dt.year
df["day"]                 = df["Date"].dt.day
df["dayofweek"]           = df["Date"].dt.dayofweek
df["dayofyear"]           = df["Date"].dt.dayofyear
df["month"]               = df["Date"].dt.month
df["days_in_month"]       = df["Date"].dt.days_in_month
df["days_from_month_end"] = df["days_in_month"] - df["day"]
print(f"  Calendar: {df.shape[1]} cols")

# Group 2: Lag & rolling features
for lag in [1, 6, 7, 14, 28, 365]:
    df[f"rev_lag_{lag}"] = df["Revenue"].shift(lag)
for w, lbl in [(7,"7"), (28,"28")]:
    _s = df["Revenue"].shift(1)
    df[f"rev_roll{lbl}_mean"] = _s.rolling(w, min_periods=1).mean()
df["rev_ewm7"]      = df["Revenue"].shift(1).ewm(span=7,  adjust=False).mean()
df["rev_ewm28"]     = df["Revenue"].shift(1).ewm(span=28, adjust=False).mean()
df["margin_lag_7"]  = df["margin_pct"].shift(7)
df["margin_lag_28"] = df["margin_pct"].shift(28)
print(f"  Lags + rolling: {df.shape[1]} cols")

In [ ]:
# Group 3: DOY seasonal priors (training data only — no leakage)
_tr = df[df["split"] == "train"].copy()
_tr = _tr.merge(sales[["Date","Revenue"]], on="Date", how="left", suffixes=("","_s"))
if "Revenue_s" in _tr.columns:
    _tr["Revenue"] = _tr["Revenue_s"].fillna(_tr["Revenue"])
    _tr.drop(columns=["Revenue_s"], inplace=True)

_doy = _tr.groupby("dayofyear")["Revenue"].agg(
    doy_rev_mean="mean", doy_rev_median="median", doy_rev_std="std").reset_index()
_mdow = _tr.groupby(["month","dayofweek"])["Revenue"].agg(month_dow_rev_mean="mean").reset_index()
_rec20 = (_tr[_tr["year"] >= 2020].groupby("dayofyear")["Revenue"]
          .agg(recent_doy_rev_mean="mean").reset_index())
_rec18 = (_tr[_tr["year"] >= 2018].groupby("dayofyear")["Revenue"]
          .agg(post2018_doy_rev_mean="mean").reset_index())
_doy_peak = (_tr[_tr["month"].isin([3,4,5,6])].groupby("dayofyear")["Revenue"]
             .agg(peak_doy_rev_mean="mean").reset_index())

df = df.merge(_doy,      on="dayofyear",           how="left")
df = df.merge(_mdow,     on=["month","dayofweek"], how="left")
df = df.merge(_rec20,    on="dayofyear",           how="left")
df = df.merge(_rec18,    on="dayofyear",           how="left")
df = df.merge(_doy_peak, on="dayofyear",           how="left")
df["peak_doy_rev_mean"]   = df["peak_doy_rev_mean"].fillna(df["doy_rev_mean"])
df["log_doy_rev_mean"]    = np.log1p(df["doy_rev_mean"])
df["log_recent_doy_mean"] = np.log1p(df["recent_doy_rev_mean"])
print(f"  DOY priors: {df.shape[1]} cols")

# Group 4: Order density
_daily_ord = orders.groupby("order_date").size().reset_index(name="n_orders")
_daily_ord["dayofyear"] = pd.to_datetime(_daily_ord["order_date"]).dt.dayofyear
_doy_ord = _daily_ord.groupby("dayofyear")["n_orders"].agg(doy_orders_mean="mean").reset_index()
df = df.merge(_doy_ord, on="dayofyear", how="left")
print(f"  Order density: {df.shape[1]} cols")

# Group 5: Tết proximity
_tet = pd.to_datetime([
    "2012-01-23","2013-02-10","2014-01-31","2015-02-19","2016-02-08",
    "2017-01-28","2018-02-16","2019-02-05","2020-01-25","2021-02-12",
    "2022-02-01","2023-01-22","2024-02-10",
])
def _days_to_tet(date):
    diffs = (_tet - date).days.values
    return int(diffs[np.abs(diffs).argmin()])
df["days_to_tet"] = df["Date"].apply(_days_to_tet)
print(f"  Tết proximity: {df.shape[1]} cols")
print(f"\n  Feature engineering complete: {df.shape[1]} total columns")

### Block 5 — Final Assembly (25-Feature Selection)

In [ ]:
FEATURES_V7 = [
    "rev_lag_1",    "rev_lag_365",  "rev_lag_7",    "rev_lag_14",
    "rev_lag_28",   "rev_lag_6",    "rev_ewm7",     "rev_ewm28",
    "rev_roll7_mean","rev_roll28_mean","doy_rev_mean","log_doy_rev_mean",
    "doy_rev_median","post2018_doy_rev_mean","peak_doy_rev_mean",
    "month_dow_rev_mean","recent_doy_rev_mean","log_recent_doy_mean",
    "doy_rev_std",  "year",         "day",          "days_from_month_end",
    "dayofweek",    "doy_orders_mean","days_to_tet",
]
MARGIN_LAG_COLS = ["margin_lag_7", "margin_lag_28"]
ALL_MODEL_COLS  = FEATURES_V7 + MARGIN_LAG_COLS

_missing = [f for f in ALL_MODEL_COLS if f not in df.columns]
if _missing:
    raise ValueError(f"Missing features: {_missing}")

train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
test_df  = df[df["split"] == "test"].copy().reset_index(drop=True)
assert len(test_df) == N_TEST

X_train        = train_df[FEATURES_V7].reset_index(drop=True)
X_test         = test_df[FEATURES_V7].reset_index(drop=True)
y_train_rev    = train_df["log_revenue"].reset_index(drop=True)
y_train_margin = train_df["margin_pct"].reset_index(drop=True)
y_train_raw    = train_df["Revenue"].reset_index(drop=True)

X_train.to_parquet(f"{OUT}/X_train_v7.parquet", index=False)
X_test.to_parquet( f"{OUT}/X_test_v7.parquet",  index=False)
train_df.to_parquet(f"{OUT}/train_df_v7.parquet", index=False)
with open(f"{OUT}/feature_list_v7.json","w") as f:
    json.dump(FEATURES_V7, f, indent=2)

print(f"  Features selected : {len(FEATURES_V7)}")
print(f"  X_train           : {X_train.shape}")
print(f"  X_test            : {X_test.shape}  ✓ {N_TEST} rows")

### Block 6 — Baseline Model & Validation Infrastructure

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import RidgeCV
import lightgbm as lgb
import optuna, joblib
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)
tscv = TimeSeriesSplit(n_splits=5, gap=30)

def compute_metrics(y_true_log, y_pred_log, label=""):
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_true_log)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1))) * 100
    if label:
        print(f"  {label:55s}  MAE={mae:>12,.0f}  RMSE={rmse:>12,.0f}  "
              f"R²={r2:.4f}  MAPE={mape:.2f}%")
    return {"mae": mae, "rmse": rmse, "r2": r2, "mape": mape}

mask_tr22 = pd.to_datetime(train_df["Date"]) < "2022-01-01"
mask_va22 = pd.to_datetime(train_df["Date"]) >= "2022-01-01"
X_tr22, X_va22 = X_train[mask_tr22], X_train[mask_va22]
y_tr22, y_va22 = y_train_rev[mask_tr22], y_train_rev[mask_va22]
X_filled   = X_train.fillna(X_train.median())
X_tr22_f   = X_filled[mask_tr22]
X_va22_f   = X_filled[mask_va22]

lgbm_base = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.05, num_leaves=63,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1)
lgbm_base.fit(X_tr22, y_tr22, eval_set=[(X_va22, y_va22)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)])
compute_metrics(y_va22.values, lgbm_base.predict(X_va22),
                label="LightGBM untuned baseline 25-feat (2022 hold-out)")

### Block 7 — LightGBM Optuna (100 Trials)

In [ ]:
def lgbm_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 3000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 15, 255),
        "max_depth":         trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42, "n_jobs": -1, "verbose": -1,
    }
    fold_rmses = []
    for tr, va in tscv.split(X_train):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_train.iloc[tr], y_train_rev.iloc[tr],
              eval_set=[(X_train.iloc[va], y_train_rev.iloc[va])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)])
        fold_rmses.append(np.sqrt(mean_squared_error(
            np.expm1(y_train_rev.iloc[va]), np.expm1(m.predict(X_train.iloc[va])))))
    return np.mean(fold_rmses)

study_lgbm = optuna.create_study(direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True))
study_lgbm.optimize(lgbm_objective, n_trials=100, show_progress_bar=True)
p_lgbm = study_lgbm.best_params
p_lgbm.update({"random_state": 42, "n_jobs": -1, "verbose": -1})
with open(f"{OUT}/best_params_lgbm_v7.json","w") as f:
    json.dump(p_lgbm, f, indent=2)
print(f"\n  Best CV RMSE: {study_lgbm.best_value:,.0f} VND")

print("\n  [Depth variants — 2022 hold-out]")
_best_lgbm_mae, _best_lgbm_params = np.inf, p_lgbm
for _lbl, _par in [
    ("Optuna original",       {**p_lgbm}),
    ("Optuna max_depth freed", {k:v for k,v in p_lgbm.items() if k != "max_depth"}),
    ("Optuna max_depth=-1",   {**p_lgbm, "max_depth": -1}),
]:
    _m = lgb.LGBMRegressor(**_par)
    _m.fit(X_tr22, y_tr22)
    _met = compute_metrics(y_va22.values, _m.predict(X_va22), label=f"  {_lbl}")
    if _met["mae"] < _best_lgbm_mae:
        _best_lgbm_mae    = _met["mae"]
        _best_lgbm_params = dict(_par)
p_lgbm = _best_lgbm_params
print(f"\n  Best LGBM → MAE={_best_lgbm_mae:,.0f}")

### Block 8 — XGBoost Optuna (100 Trials)

In [ ]:
def xgb_objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 200, 3000),
        "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma":            trial.suggest_float("gamma", 1e-8, 5.0, log=True),
        "random_state": 42, "n_jobs": -1, "verbosity": 0, "tree_method": "hist",
    }
    fold_rmses = []
    for tr, va in tscv.split(X_train):
        m = XGBRegressor(**params)
        m.fit(X_train.iloc[tr], y_train_rev.iloc[tr],
              eval_set=[(X_train.iloc[va], y_train_rev.iloc[va])], verbose=False)
        fold_rmses.append(np.sqrt(mean_squared_error(
            np.expm1(y_train_rev.iloc[va]), np.expm1(m.predict(X_train.iloc[va])))))
    return np.mean(fold_rmses)

study_xgb = optuna.create_study(direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True))
study_xgb.optimize(xgb_objective, n_trials=100, show_progress_bar=True)
p_xgb = study_xgb.best_params
p_xgb.update({"random_state": 42, "n_jobs": -1, "verbosity": 0, "tree_method": "hist"})
with open(f"{OUT}/best_params_xgb_v7.json","w") as f:
    json.dump(p_xgb, f, indent=2)
print(f"  Best CV RMSE: {study_xgb.best_value:,.0f} VND")

xgb_ho = XGBRegressor(**p_xgb)
xgb_ho.fit(X_tr22, y_tr22, eval_set=[(X_va22, y_va22)], verbose=False)
compute_metrics(y_va22.values, xgb_ho.predict(X_va22), label="  XGBoost tuned (2022 hold-out)")

### Block 9 — CatBoost Optuna (100 Trials)

In [ ]:
def cat_objective(trial):
    params = {
        "iterations":          trial.suggest_int("iterations", 200, 3000),
        "learning_rate":       trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "depth":               trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg":         trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":     trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "border_count":        trial.suggest_int("border_count", 32, 254),
        "random_seed": 42, "verbose": 0, "allow_writing_files": False,
    }
    fold_rmses = []
    for tr, va in tscv.split(X_filled):
        m = CatBoostRegressor(**params)
        m.fit(X_filled.iloc[tr], y_train_rev.iloc[tr],
              eval_set=(X_filled.iloc[va], y_train_rev.iloc[va]), early_stopping_rounds=50)
        fold_rmses.append(np.sqrt(mean_squared_error(
            np.expm1(y_train_rev.iloc[va]), np.expm1(m.predict(X_filled.iloc[va])))))
    return np.mean(fold_rmses)

study_cat = optuna.create_study(direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True))
study_cat.optimize(cat_objective, n_trials=100, show_progress_bar=True)
p_cat = study_cat.best_params
p_cat.update({"random_seed": 42, "verbose": 0, "allow_writing_files": False})
with open(f"{OUT}/best_params_cat_v7.json","w") as f:
    json.dump(p_cat, f, indent=2)
print(f"  Best CV RMSE: {study_cat.best_value:,.0f} VND")

cat_ho = CatBoostRegressor(**p_cat)
cat_ho.fit(X_tr22_f, y_tr22, eval_set=(X_va22_f, y_va22), early_stopping_rounds=50)
compute_metrics(y_va22.values, cat_ho.predict(X_va22_f), label="  CatBoost tuned (2022 hold-out)")

### Block 10 — Margin Model (100 Trials)

In [ ]:
_p1          = y_train_margin.quantile(0.01)
y_mg_clipped = y_train_margin.clip(lower=_p1)

X_train_mg = train_df[FEATURES_V7 + MARGIN_LAG_COLS].reset_index(drop=True)
X_tr22_mg  = X_train_mg[mask_tr22]
X_va22_mg  = X_train_mg[mask_va22]
X_test_mg  = test_df[FEATURES_V7 + MARGIN_LAG_COLS].reset_index(drop=True)

def margin_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 2000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 15, 127),
        "max_depth":         trial.suggest_int("max_depth", 3, 8),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        "random_state": 42, "n_jobs": -1, "verbose": -1,
    }
    fold_maes = []
    for tr, va in tscv.split(X_train_mg):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_train_mg.iloc[tr], y_mg_clipped.iloc[tr],
              eval_set=[(X_train_mg.iloc[va], y_mg_clipped.iloc[va])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)])
        fold_maes.append(mean_absolute_error(y_mg_clipped.iloc[va], m.predict(X_train_mg.iloc[va])))
    return np.mean(fold_maes)

study_mg = optuna.create_study(direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True))
study_mg.optimize(margin_objective, n_trials=100, show_progress_bar=True)
p_mg = study_mg.best_params
p_mg.update({"random_state": 42, "n_jobs": -1, "verbose": -1})
with open(f"{OUT}/best_params_margin_v7.json","w") as f:
    json.dump(p_mg, f, indent=2)
print(f"  Margin best CV MAE: {study_mg.best_value:.4f}%")

mgm_ho = lgb.LGBMRegressor(**p_mg)
mgm_ho.fit(X_tr22_mg, y_mg_clipped[mask_tr22])
_pm   = mgm_ho.predict(X_va22_mg)
_pm_c = np.clip(_pm, 2.0, 25.0)
print(f"  Margin 2022 hold-out: MAE={mean_absolute_error(y_train_margin[mask_va22], _pm_c):.4f}%  "
      f"R²={r2_score(y_train_margin[mask_va22], _pm_c):.4f}")

### Block 11 — Ridge Stack Meta-Learner (3-Model Ensemble)

In [ ]:
lgbm_v = lgb.LGBMRegressor(**p_lgbm); lgbm_v.fit(X_tr22, y_tr22)
xgb_v  = XGBRegressor(**p_xgb);       xgb_v.fit(X_tr22, y_tr22)
cat_v  = CatBoostRegressor(**p_cat)
cat_v.fit(X_tr22_f, y_tr22, eval_set=(X_va22_f, y_va22), early_stopping_rounds=50)

p_l = lgbm_v.predict(X_va22)
p_x = xgb_v.predict(X_va22)
p_c = cat_v.predict(X_va22_f)
y_22 = y_va22.values
L_22 = np.column_stack([p_l, p_x, p_c])

print("\n  [Individual model 2022 hold-out]")
compute_metrics(y_22, p_l, label="  LightGBM tuned")
compute_metrics(y_22, p_x, label="  XGBoost  tuned")
compute_metrics(y_22, p_c, label="  CatBoost tuned")

ridge = RidgeCV(alphas=np.logspace(-3,3,50), cv=10, fit_intercept=True)
ridge.fit(L_22, y_22)
pred_stack = ridge.predict(L_22)
m_stack    = compute_metrics(y_22, pred_stack, label="  STACK Ridge (3-model)")
print(f"\n  Ridge weights → LGBM={ridge.coef_[0]:.3f}  XGB={ridge.coef_[1]:.3f}  "
      f"CAT={ridge.coef_[2]:.3f}  α={ridge.alpha_:.4f}")
joblib.dump(ridge, f"{MODELS}/ridge_v7.pkl")

### Block 12 — Full Refit on All Training Data

In [ ]:
lgbm_full = lgb.LGBMRegressor(**p_lgbm)
lgbm_full.fit(X_train, y_train_rev)
joblib.dump(lgbm_full, f"{MODELS}/lgbm_v7.pkl"); print("  LightGBM ✓")

xgb_full = XGBRegressor(**p_xgb)
xgb_full.fit(X_train, y_train_rev)
joblib.dump(xgb_full,  f"{MODELS}/xgb_v7.pkl");  print("  XGBoost  ✓")

cat_full = CatBoostRegressor(**p_cat)
cat_full.fit(X_filled, y_train_rev)
joblib.dump(cat_full,  f"{MODELS}/cat_v7.pkl");   print("  CatBoost ✓")

margin_mdl = lgb.LGBMRegressor(**p_mg)
margin_mdl.fit(X_train_mg, y_mg_clipped)
joblib.dump(margin_mdl, f"{MODELS}/margin_v7.pkl"); print("  Margin   ✓")

### Block 13 — Recursive Forecast (548 Test Days)

In [ ]:
TEST_DATES       = pd.to_datetime(sample_sub["Date"].values)
assert len(TEST_DATES) == N_TEST

X_test_filled    = X_test.fillna(X_test.median())
X_test_mg_filled = X_test_mg.fillna(X_test_mg.median())
train_cap        = y_train_raw.quantile(0.999)
LOG_CAP          = np.log1p(train_cap)

train_df["Date"] = pd.to_datetime(train_df["Date"])
rev_buffer       = dict(zip(train_df["Date"], train_df["log_revenue"]))
margin_buffer    = dict(zip(train_df["Date"], train_df["margin_pct"]))

REV_LAG_COLS         = [c for c in FEATURES_V7 if "rev_lag_" in c or "rev_roll" in c or "rev_ewm" in c]
MARGIN_LAG_COLS_FILL = list(MARGIN_LAG_COLS)

def _get_lag(buf, date, lag):
    return buf.get(date - pd.Timedelta(days=lag), np.nan)
def _get_roll_mean(buf, date, w):
    vals  = [buf.get(date - pd.Timedelta(days=i), np.nan) for i in range(1, w+1)]
    valid = [v for v in vals if not np.isnan(v)]
    return np.mean(valid) if valid else np.nan
def _get_ewm(buf, date, span):
    alpha = 2/(span+1); result, found = None, False
    for i in range(1, span*5):
        v = buf.get(date - pd.Timedelta(days=i), None)
        if v is not None:
            result = v if not found else alpha*v+(1-alpha)*result
            found  = True
    return result if found else np.nan

pred_log_rev = np.zeros(len(TEST_DATES))
pred_margin  = np.zeros(len(TEST_DATES))

print(f"  Forecasting {len(TEST_DATES)} days ({TEST_DATES[0].date()} → {TEST_DATES[-1].date()})...")

for i, date in enumerate(TEST_DATES):
    row    = X_test_filled.iloc[i].copy()
    row_mg = X_test_mg_filled.iloc[i].copy()

    for col in REV_LAG_COLS:
        if "rev_lag_" in col:
            _val = _get_lag(rev_buffer, date, int(col.replace("rev_lag_","")))
        elif "rev_roll7_mean"  in col:
            _val = _get_roll_mean(rev_buffer, date, 7)
        elif "rev_roll28_mean" in col:
            _val = _get_roll_mean(rev_buffer, date, 28)
        elif "rev_ewm7"        in col:
            _val = _get_ewm(rev_buffer, date, 7)
        elif "rev_ewm28"       in col:
            _val = _get_ewm(rev_buffer, date, 28)
        else:
            _val = np.nan
        row[col] = _val; row_mg[col] = _val

    for col in MARGIN_LAG_COLS_FILL:
        lag = int(col.replace("margin_lag_",""))
        row_mg[col] = margin_buffer.get(date - pd.Timedelta(days=lag), y_train_margin.median())

    X_row   = pd.DataFrame([row])
    X_row_f = X_row.fillna(X_row.median())
    X_mg    = pd.DataFrame([row_mg])

    p_l = lgbm_full.predict(X_row)[0]
    p_x = xgb_full.predict(X_row)[0]
    p_c = cat_full.predict(X_row_f)[0]

    p_stack  = ridge.predict(np.array([[p_l, p_x, p_c]]))[0]
    p_stack  = np.clip(p_stack, 0.0, LOG_CAP)
    p_margin = margin_mdl.predict(X_mg)[0]
    p_margin_clipped_i = np.clip(p_margin, 2.0, 25.0)

    pred_log_rev[i] = p_stack
    pred_margin[i]  = p_margin
    rev_buffer[date]    = p_stack
    margin_buffer[date] = p_margin_clipped_i

    if (i+1) % 100 == 0 or i == 0 or i == len(TEST_DATES) - 1:
        print(f"    Day {i+1:3d} ({date.date()})  rev={np.expm1(p_stack):>12,.0f}  "
              f"margin={p_margin_clipped_i:.2f}%")

pred_margin_clipped = np.clip(pred_margin, 2.0, 25.0)
pred_revenue        = np.expm1(pred_log_rev)
pred_cogs           = np.maximum(pred_revenue * (1 - pred_margin_clipped/100), 0)
implied_margin      = (pred_revenue - pred_cogs) / pred_revenue * 100

print(f"\n  Forecast summary:")
print(f"    Revenue : min={pred_revenue.min():>12,.0f}  mean={pred_revenue.mean():>12,.0f}  max={pred_revenue.max():>12,.0f}")
print(f"    COGS    : min={pred_cogs.min():>12,.0f}  mean={pred_cogs.mean():>12,.0f}  max={pred_cogs.max():>12,.0f}")
print(f"    Margin  : mean={implied_margin.mean():.2f}%  min={implied_margin.min():.2f}%  max={implied_margin.max():.2f}%")
print(f"    COGS > Revenue: {(pred_cogs > pred_revenue).sum()} (expect 0)")

### Block 14 — Write `submission_final.csv` + Validity Checks

In [ ]:
_sub_dates = pd.to_datetime(TEST_DATES)
submission = pd.DataFrame({
    "Date":    _sub_dates.strftime("%Y-%m-%d"),
    "Revenue": np.round(pred_revenue, 2),
    "COGS":    np.round(pred_cogs,    2),
})
out_path = f"{SUBMISSION}/submission_final.csv"
submission.to_csv(out_path, index=False)

assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == N_TEST
assert submission["Revenue"].isna().sum() == 0
assert submission["COGS"].isna().sum() == 0
assert (submission["Revenue"] > 0).all()
assert (submission["COGS"]    > 0).all()
assert (submission["COGS"] <= submission["Revenue"]).all()

with open(out_path,"rb") as fh:
    sha = hashlib.sha256(fh.read()).hexdigest()

print(f"  ✓ Rows          : {len(submission)}  (expected: {N_TEST})")
print(f"  ✓ No NaNs")
print(f"  ✓ All Revenue > 0  |  All COGS > 0")
print(f"  ✓ COGS ≤ Revenue on all rows")
print(f"  SHA256: {sha}")
print(f"\n  First 5 rows:\n{submission.head(5).to_string(index=False)}")
print(f"\n  Last  5 rows:\n{submission.tail(5).to_string(index=False)}")

### Block 15 — SHAP Feature Importance

In [ ]:
import shap

print("  Computing SHAP values on XGBoost full model...")
explainer   = shap.TreeExplainer(xgb_full)
shap_values = explainer.shap_values(X_train)

shap_importance = pd.DataFrame({
    "feature":       FEATURES_V7,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print(f"\n  All 25 features by mean |SHAP|:")
print(shap_importance.to_string(index=False))
shap_importance.to_csv(f"{OUT}/shap_importance_v7.csv", index=False)

shap.summary_plot(shap_values, X_train, feature_names=FEATURES_V7,
                  plot_type="bar", max_display=25, show=False)
plt.tight_layout()
plt.savefig(f"{OUT}/shap_summary_v7.png", dpi=150, bbox_inches="tight")
plt.show()

_biz = {
    "rev_lag_1":             "Yesterday's revenue — strongest single predictor",
    "rev_lag_7":             "Same weekday last week — weekly rhythm",
    "rev_lag_14":            "Two weeks ago — medium-term momentum",
    "rev_lag_28":            "Monthly cycle anchor",
    "rev_lag_365":           "Same day last year — YoY anchor",
    "rev_lag_6":             "6-day lag (PACF=0.305 — direct autocorrelation)",
    "rev_ewm7":              "Exponentially weighted trend (7-day)",
    "rev_ewm28":             "Exponentially weighted trend (28-day)",
    "rev_roll7_mean":        "7-day rolling average — short-term level",
    "rev_roll28_mean":       "28-day rolling average — medium-term level",
    "doy_rev_mean":          "Historical average for this day of year",
    "log_doy_rev_mean":      "Log-scale DOY prior",
    "doy_rev_median":        "Robust DOY central tendency",
    "post2018_doy_rev_mean": "Recent-regime DOY average (post-2018)",
    "peak_doy_rev_mean":     "Peak-season DOY average (Mar–Jun)",
    "month_dow_rev_mean":    "Month × weekday interaction prior",
    "recent_doy_rev_mean":   "Post-2020 DOY average — most recent regime",
    "log_recent_doy_mean":   "Log-scale recent DOY prior",
    "doy_rev_std":           "DOY volatility — signals uncertain days",
    "year":                  "Long-term revenue trend direction",
    "day":                   "Day of month — salary cycle effects",
    "days_from_month_end":   "Proximity to month end — salary spike signal",
    "dayofweek":             "Day of week seasonality",
    "doy_orders_mean":       "Historical order volume for this DOY",
    "days_to_tet":           "Proximity to Tết — biggest seasonal driver",
}

print(f"\n  Business interpretation of top 10 drivers:")
for i, row in shap_importance.head(10).iterrows():
    print(f"    {i+1:>2}. {row['feature']:<30s}  SHAP={row['mean_abs_shap']:>10.4f}  "
          f"{_biz.get(row['feature'], '')}")

print(f"\n  ✅ PIPELINE COMPLETE")
print(f"     Stack MAE : {m_stack['mae']:,.0f}")
print(f"     Stack R²  : {m_stack['r2']:.4f}")
print(f"     Output    : submission_final.csv  ← submit this")